# Rural primary-care workforce and infant mortality

This notebook matches RBI Table 16 (2023 rural PHC doctor staffing) with RBI Table 4 (2023 infant mortality rate). The staffing ratio is **doctors in position divided by the source-reported required doctors**. The requirement is an administrative PHC staffing benchmark (approximately one doctor per PHC), not a population-based estimate of how many doctors residents need. A ratio of 2 therefore means about twice the benchmark staffing count, not twice the population's required doctor coverage.

The analysis is cross-sectional and observational; it is not a causal estimate.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

DATA = Path('../Dataset')
staff = pd.read_csv(DATA / 'state_rural_phc_doctors_2023.csv')
imr = pd.read_csv(DATA / 'state_infant_mortality_rate_2023.csv')
staff.head()

## 1. Validate and harmonise state labels

The RBI workforce table combines Dadra & Nagar Haveli and Daman & Diu; the IMR table labels the same combined observation as Dadra & Nagar Haveli (includes Daman & Diu). Missing source markers remain missing, and the all-India aggregate is excluded.

In [ ]:
## 2. Interpreting the staffing benchmark

A ratio of 1.0 means in-position doctors equal the source-reported requirement; below 1.0 means fewer doctors are in position than required; above 1.0 means staffing exceeds that administrative benchmark. Because the denominator is a count of required PHC posts (approximately one doctor per PHC), the ratio is **not** doctors per capita, population coverage, or proof of adequate access. Small denominators can create very high ratios: Delhi has 5 required doctors and 16 in position (3.20); Goa has 21 required and 97 in position (4.62).

valid = df.dropna(subset=['phc_doctor_coverage_ratio']).copy()
valid = valid.sort_values('phc_doctor_coverage_ratio')
display(valid[['state_ut','required_rural_phc_doctors','in_position_rural_phc_doctors','phc_doctor_coverage_ratio']].round(3))
print('Median staffing ratio:', round(valid.phc_doctor_coverage_ratio.median(), 3))
print('Below reported requirement:', int((valid.phc_doctor_coverage_ratio < 1).sum()), 'of', len(valid))
print('At or above reported requirement:', int((valid.phc_doctor_coverage_ratio >= 1).sum()), 'of', len(valid))

## 3. Small-denominator sensitivity

To assess whether very small PHC requirement counts dominate the association, the notebook reports the full sample and a restricted sample excluding observations with fewer than 50 required rural PHC doctors. This is a sensitivity check, not a claim that the threshold is a formal statistical rule. The 50-post cutoff is chosen transparently to reduce volatility in very small denominators.

In [ ]:
from scipy.stats import pearsonr, spearmanr

def association_summary(data, label):
    x = data['phc_doctor_coverage_ratio']
    y = data['infant_mortality_rate_2023_per_1000_live_births']
    pr, pp = pearsonr(x, y)
    sr, sp = spearmanr(x, y)
    print(f'{label}: n={len(data)}')
    print(f'  Median staffing ratio: {data.phc_doctor_coverage_ratio.median():.3f}')
    print(f'  Pearson r={pr:.3f}, p={pp:.4f}')
    print(f'  Spearman rho={sr:.3f}, p={sp:.4f}')

analysis = df.dropna(subset=['phc_doctor_coverage_ratio']).copy()
association_summary(analysis, 'All matched states/UTs')
large_denominator = analysis.loc[analysis['required_rural_phc_doctors'] >= 50].copy()
association_summary(large_denominator, 'Restricted sample: required doctors >= 50')

In [ ]:
valid = df.dropna(subset=['phc_doctor_coverage_ratio']).copy()
valid = valid.sort_values('phc_doctor_coverage_ratio')
display(valid[['state_ut','required_rural_phc_doctors','in_position_rural_phc_doctors','phc_doctor_coverage_ratio']].round(3))
print('Median staffing coverage ratio:', round(valid.phc_doctor_coverage_ratio.median(), 3))
print('States/UTs below reported requirement:', int((valid.phc_doctor_coverage_ratio < 1).sum()), 'of', len(valid))
print('States/UTs at or above reported requirement:', int((valid.phc_doctor_coverage_ratio >= 1).sum()), 'of', len(valid))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(valid.state_ut, valid.phc_doctor_coverage_ratio)
ax.axvline(1, linestyle='--', linewidth=1, color='black', label='Reported requirement met')
ax.set_xlabel('Doctors in position / required doctors')
ax.set_title('Rural PHC doctor staffing relative to reported requirement, 2023')
ax.legend()
fig.tight_layout()
plt.show()

## 3. Staffing coverage and infant mortality

The two indicators refer to 2023. Pearson correlation measures linear association; Spearman correlation measures rank association and is less sensitive to the exact scale. Neither controls for socioeconomic determinants or establishes causality.

In [ ]:
analysis = df.dropna(subset=['phc_doctor_coverage_ratio']).copy()
x = analysis['phc_doctor_coverage_ratio']
y = analysis['infant_mortality_rate_2023_per_1000_live_births']
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, alpha=0.8)
for _, row in analysis.iterrows():
    ax.annotate(row['state_ut'], (row['phc_doctor_coverage_ratio'], row['infant_mortality_rate_2023_per_1000_live_births']), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.axvline(1, linestyle='--', linewidth=1, color='black', label='Reported requirement met')
ax.set_xlabel('In-position rural PHC doctors / required rural PHC doctors, 2023')
ax.set_ylabel('Infant mortality rate per 1,000 live births, 2023')
ax.set_title('Rural PHC staffing benchmark and infant mortality')
ax.legend()
fig.tight_layout()
plt.show()

## 5. Interpretation and limitations

In the full sample (34 observations), Pearson r is approximately -0.538 and Spearman rho is approximately -0.562. In the restricted sample with at least 50 required doctors (27 observations), Pearson r is approximately -0.498 and Spearman rho is approximately -0.368. The negative linear association remains, but the rank association weakens and is no longer statistically significant at the 5% level in the restricted sample. This shows that small-denominator observations influence the strength of the rank relationship, though they do not explain the entire negative linear association.

This is a bivariate association, not a causal effect. The ratio is measured against the administrative PHC staffing requirement, not population need. It excludes private providers, urban facilities, nurses, specialists outside PHCs and service quality. IMR is also affected by income, maternal education, nutrition, sanitation, fertility, transport and other determinants. The analysis is cross-sectional, and small states/UTs may have volatile rates. Missing source markers remain missing and are not imputed as zero.

A stronger next step is an adjusted model with socioeconomic controls and, ideally, a multi-year state panel with consistent definitions.

In [ ]:
# Reproducibility checks
assert df.state_key.is_unique
assert len(analysis) == 34
assert len(large_denominator) == 27
assert analysis['phc_doctor_coverage_ratio'].notna().all()
assert analysis['infant_mortality_rate_2023_per_1000_live_births'].notna().all()
print('Notebook checks passed.')